# Backtranslation (BT) Sensitivity Analysis
This notebook tests how **evaluation metrics** and **model rankings** change when you switch the **backtranslation (BT)** system used to translate non‑English summaries into English.

**Hypothesis.** If a metric is sensitive and reliable, then as we use better backtranslation models, the scores from that metric should increase accordingly. If scores are flat, inconsistent, or noisy across clearly better models, the metric might not be trustworthy.

> This notebook expects a single CSV that already contains outputs scored under **multiple `bt_model` values**.


In [ ]:
# ==== Configuration ====
INPUT_CSV = "/Users/madhurimachakraborty/Documents/GitHub/CodeClarity/data/evalScores0.7.csv"  # <--- set if needed

# Metric columns present in your CSV
METRICS = [
    'bertscore_f1','bertscore_precision','bertscore_recall',
    'bleu','chrf++','rougeL','meteor','comet','side'
]

# Columns that should exist
REQUIRED_COLS = ['bt_model','model_name','bt_language']
OPTIONAL_ID_COLS = ['sample_id']  # used for per-sample reliability, if available


In [ ]:
# ==== Imports ====
import math
import numpy as np
import pandas as pd
from itertools import combinations
from scipy.stats import kendalltau, spearmanr
import matplotlib.pyplot as plt

# Matplotlib defaults (no style/color forcing)
plt.rcParams.update({'figure.dpi': 110}
)

In [ ]:
# ==== Load & sanitize ====
df = pd.read_csv(INPUT_CSV)

missing = [c for c in REQUIRED_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Coerce metric columns to numeric
for col in METRICS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("Shape:", df.shape)
print("BT models:", sorted(df['bt_model'].dropna().unique().tolist()))
print("Models:", sorted(df['model_name'].dropna().unique().tolist()))
df.head(3)

## 1) Absolute score shifts by BT system
Mean score of each metric per `bt_model`. Large spreads indicate strong BT sensitivity of absolute scores.


In [ ]:
# Mean per BT system
bt_means = df.groupby('bt_model')[METRICS].mean().sort_index()
bt_means

In [ ]:
# Heatmap (matplotlib only)
def heatmap(df_matrix, title):
    import math
    import numpy as np
    import matplotlib.pyplot as plt
    vals = df_matrix.values
    fig, ax = plt.subplots(figsize=(10, 5 + 0.25*len(df_matrix)))
    im = ax.imshow(vals, aspect='auto')
    ax.set_title(title)
    ax.set_xticks(range(df_matrix.shape[1]))
    ax.set_xticklabels(list(df_matrix.columns), rotation=45, ha='right')
    ax.set_yticks(range(df_matrix.shape[0]))
    ax.set_yticklabels(list(df_matrix.index))
    # annotations
    for i in range(df_matrix.shape[0]):
        for j in range(df_matrix.shape[1]):
            v = vals[i, j]
            if isinstance(v, (float, np.floating)) and not math.isnan(v):
                txt = f"{v:.3f}"
            else:
                txt = "NA"
            ax.text(j, i, txt, ha='center', va='center', fontsize=8)
    fig.colorbar(im, ax=ax, shrink=0.8)
    fig.tight_layout()
    plt.show()

heatmap(bt_means, "Mean metric scores by backtranslation model")

## 2) Ranking stability across BT systems
For each metric:
- Compute per-`bt_model` **mean scores** per `model_name`.
- Rank models (higher is better).
- Summarize **Kendall’s τ** across all BT pairs and **Top‑1 winner agreement**.


In [ ]:
def rank_table(df, metric):
    tab = df.pivot_table(index='model_name', columns='bt_model', values=metric, aggfunc='mean')
    ranks = tab.rank(ascending=False, method='average')
    return tab, ranks

def ranking_stability(df, metric):
    _, ranks = rank_table(df, metric)
    taus = []
    cols = [c for c in ranks.columns if c is not None]
    for a, b in combinations(cols, 2):
        ra, rb = ranks[a], ranks[b]
        mask = ra.notna() & rb.notna()
        if mask.sum() >= 2:
            tau, _ = kendalltau(ra[mask], rb[mask], nan_policy='omit')
            taus.append(tau)
    tau_mean = float(np.nanmean(taus)) if len(taus) else float('nan')
    # Top-1 winner agreement
    winner = ranks.idxmin(axis=0)  # rank 1.0 is min
    top1_agreement = winner.value_counts(normalize=True).max() if not winner.empty else float('nan')
    return tau_mean, top1_agreement, ranks

rows = []
for m in METRICS:
    if m not in df.columns: 
        continue
    tau_mean, top1_agree, ranks = ranking_stability(df, m)
    rows.append({'metric': m, 'kendall_tau_mean': tau_mean, 'top1_agreement': top1_agree})
rank_stab = pd.DataFrame(rows).sort_values('kendall_tau_mean', ascending=False)
rank_stab

In [ ]:
# Bar plots (matplotlib only)
def barplot_from_series(series, title, ylabel):
    import numpy as np
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(series))
    ax.bar(x, series.values)
    ax.set_xticks(x)
    ax.set_xticklabels(series.index, rotation=45, ha='right')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    plt.show()

barplot_from_series(rank_stab.set_index('metric')['kendall_tau_mean'],
                    'Kendall\'s τ (avg across BT pairs) by metric', 'τ')
barplot_from_series(rank_stab.set_index('metric')['top1_agreement'],
                    'Top‑1 winner agreement across BTs by metric', 'agreement (0..1)')

## 3) Pairwise flip rate across BTs
How often does the order between two models change when switching BT systems?


In [ ]:
def pairwise_flip_rate(df, metric, atol=1e-6):
    tab = df.pivot_table(index='model_name', columns='bt_model', values=metric, aggfunc='mean')
    models = tab.index.tolist()
    flips = []
    for i, j in combinations(range(len(models)), 2):
        diff = tab.iloc[i] - tab.iloc[j]  # vector over BTs
        d = diff.dropna().values
        if len(d) <= 1:
            continue
        signs = np.sign(d.copy())
        signs[np.isclose(d, 0, atol=atol)] = 0  # ignore tiny ties
        s = signs[signs != 0]
        if len(s) > 1:
            flips.append(int(np.any(s != s[0])))
        else:
            flips.append(0)
    return float(np.mean(flips)) if flips else float('nan')

flip_rows = []
for m in METRICS:
    if m in df.columns:
        flip_rows.append({'metric': m, 'flip_rate': pairwise_flip_rate(df, m)})
flip_table = pd.DataFrame(flip_rows).sort_values('flip_rate')
flip_table

In [ ]:
barplot_from_series(flip_table.set_index('metric')['flip_rate'],
                    'Pairwise flip rate across BTs by metric', 'flip rate (0..1)')

## 4) Per-sample reliability across BTs (if `sample_id` is available)
Mean Spearman correlation of scores for the **same sample+model+language** across BT pairs.


In [ ]:
id_cols = [c for c in ['sample_id','model_name','bt_language'] if c in df.columns]

def per_sample_bt_corr(df, metric):
    if not all(c in df.columns for c in id_cols):
        return float('nan')
    piv = df.pivot_table(index=id_cols, columns='bt_model', values=metric, aggfunc='mean')
    vals = []
    for a, b in combinations(piv.columns, 2):
        v = piv[[a, b]].dropna()
        if len(v) >= 8:
            rho, _ = spearmanr(v[a], v[b])
            vals.append(rho)
    return float(np.nanmean(vals)) if vals else float('nan')

ps_rows = []
for m in METRICS:
    if m in df.columns:
        ps_rows.append({'metric': m, 'per_sample_spearman': per_sample_bt_corr(df, m)})
ps_table = pd.DataFrame(ps_rows).sort_values('per_sample_spearman', ascending=False)
ps_table

In [ ]:
barplot_from_series(ps_table.set_index('metric')['per_sample_spearman'].fillna(0.0),
                    'Per-sample Spearman across BTs by metric', 'ρ (0..1)')

## 5) BT effect by language
Heatmaps for a form metric (BLEU) and a meaning metric (COMET).


In [ ]:
def lang_bt_matrix(df, metric):
    mat = df.groupby(['bt_language','bt_model'])[metric].mean().unstack('bt_model').sort_index()
    return mat

for metric in ['bleu','comet']:
    if metric in df.columns:
        mat = lang_bt_matrix(df, metric)
        heatmap(mat, f"{metric.upper()} by language × BT model")

## 6) Summary & export
- **Absolute scores** vary by BT system (see heatmap above).
- **Ranking stability**: higher Kendall’s τ and Top‑1 agreement ⇒ rankings less BT‑dependent.
- **Flip rates**: how often pairwise model decisions change with BT (lower is better).
- **Per‑sample Spearman**: consistency for the same items across BTs.
- **By‑language heatmaps**: where BT choice matters most.


In [ ]:
# Save summary CSVs
outdir = 'bt_sensitivity_results'
os.makedirs(outdir, exist_ok=True)
bt_means.to_csv(f'{outdir}/bt_means.csv')
rank_stab.to_csv(f'{outdir}/ranking_stability.csv', index=False)
flip_table.to_csv(f'{outdir}/flip_rates.csv', index=False)
ps_table.to_csv(f'{outdir}/per_sample_spearman.csv', index=False)
print('Saved →', [f'{outdir}/bt_means.csv', f'{outdir}/ranking_stability.csv', f'{outdir}/flip_rates.csv', f'{outdir}/per_sample_spearman.csv'])